# Continual Learning: EWC, Synaptic Intelligence, LwF, and Dark Experience Replay

This notebook explores methods that prevent catastrophic forgetting across sequential task distributions:
1. **EWC (Elastic Weight Consolidation)**: Curvature-aware parameter consolidation using diagonal Fisher Information.
2. **Synaptic Intelligence**: Online path-integral parameter importance tracking during training.
3. **LwF (Learning without Forgetting)**: Functional output regularization via knowledge distillation.
4. **DER++ (Dark Experience Replay)**: Experience replay leveraging historical un-normalized logits.

**Decision Boundary (When NOT to Use)**:
- Do not use static EWC when task sequences are very long ($T > 20$) without running-average Fisher decay, as parameter penalties accumulate and completely freeze the model.
- Do not use replay-based methods when strict data-privacy regulations (e.g., GDPR right-to-be-forgotten, patient health data) forbid caching past observations in a rehearsal memory buffer.

**Topic Bridges & Connects To**:
- `04_meta_learning_prototypical_maml_and_reptile` (Rapid task adaptation versus retention)
- `06_federated_learning_fedavg_fedprox_scaffold_and_secagg` (Distributed continual and non-IID learning)

## 2. Mathematical Formulations & Theoretical Foundations

- **EWC Quadratic Parameter Penalty**:
  $$\mathcal{L}_{\text{EWC}}(\theta) = \frac{\lambda}{2} \sum_{i=1}^D F_i (\theta_i - \theta_{A, i}^*)^2, \quad \mathcal{O}(D)$$
- **Synaptic Intelligence Importance Metric**:
  $$\Omega_k = \frac{\max(0, \omega_k)}{(\Delta \theta_k)^2 + \xi}, \quad \mathcal{O}(D)$$
- **LwF Distillation Loss**:
  $$\mathcal{L}_{\text{LwF}} = T^2 \cdot \text{KL}\left( \sigma(f_{\text{old}}(x)/T) \parallel \sigma(f_{\text{new}}(x)/T) \right), \quad \mathcal{O}(B \cdot K)$$
- **DER++ Composite Replay Loss**:
  $$\mathcal{L}_{\text{DER++}} = \alpha \|f_\theta(x_r) - z_r\|_2^2 + \beta \text{CE}(f_\theta(x_r), y_r), \quad \mathcal{O}(B_r \cdot K)$$

In [ ]:
import math
from pathlib import Path
import torch
import torch.nn.functional as F
from safetensors.torch import load_file

torch.manual_seed(42)
REPO_ROOT = next(p for p in Path.cwd().parents if (p / "Cargo.toml").exists()) if not (Path.cwd() / "Cargo.toml").exists() else Path.cwd()
GOLDEN_PATH = REPO_ROOT / "weights" / "08_learning_paradigms_rl_and_alignment" / "01_self_semi_meta_continual_federated" / "05_continual_learning_ewc_synaptic_intelligence_lwf_and_replay.safetensors"
golden = load_file(str(GOLDEN_PATH))
print(f"Loaded {len(golden)} tensors from golden checkpoint.")

In [ ]:
# 1. Elastic Weight Consolidation (EWC) Quadratic Penalty
def ewc_penalty(
    theta: torch.Tensor,
    theta_star: torch.Tensor,
    fisher_diag: torch.Tensor,
    lambda_ewc: float = 10.0,
) -> torch.Tensor:
    penalty = 0.5 * lambda_ewc * torch.sum(fisher_diag * (theta - theta_star) ** 2)
    return penalty.unsqueeze(0)

ewc_calc = ewc_penalty(golden["theta"], golden["theta_star"], golden["fisher_diag"], lambda_ewc=10.0)
diff_ewc = torch.abs(ewc_calc - golden["ewc_penalty_ref"]).item()
print(f"EWC Penalty max abs error: {diff_ewc:.2e}")
assert diff_ewc < 1e-5

In [ ]:
# 2. Synaptic Intelligence Parameter Importance
def synaptic_intelligence_omega(
    delta_theta: torch.Tensor,
    path_integral: torch.Tensor,
    xi: float = 0.1,
) -> torch.Tensor:
    return torch.clamp(path_integral, min=0.0) / (delta_theta ** 2 + xi)

si_calc = synaptic_intelligence_omega(golden["delta_theta"], golden["path_integral"], xi=0.1)
diff_si = torch.max(torch.abs(si_calc - golden["si_omega_ref"])).item()
print(f"Synaptic Intelligence Omega max abs error: {diff_si:.2e}")
assert diff_si < 1e-5

In [ ]:
# 3. Learning without Forgetting (LwF) Distillation Loss
def lwf_distillation_loss(
    new_logits: torch.Tensor,
    old_logits: torch.Tensor,
    temperature: float = 2.0,
) -> torch.Tensor:
    p_old = F.softmax(old_logits / temperature, dim=-1)
    log_p_new = F.log_softmax(new_logits / temperature, dim=-1)
    log_p_old = F.log_softmax(old_logits / temperature, dim=-1)
    kl = (p_old * (log_p_old - log_p_new)).sum(dim=-1).mean()
    return (temperature * temperature * kl).unsqueeze(0)

lwf_calc = lwf_distillation_loss(golden["new_logits"], golden["old_logits"], temperature=2.0)
diff_lwf = torch.abs(lwf_calc - golden["lwf_loss_ref"]).item()
print(f"LwF Distillation Loss max abs error: {diff_lwf:.2e}")
assert diff_lwf < 1e-5

In [ ]:
# 4. Dark Experience Replay (DER++) & Anti-Pattern Contrast
def der_dark_replay_loss(
    current_logits: torch.Tensor,
    past_logits: torch.Tensor,
    labels: torch.Tensor,
    alpha: float = 0.5,
    beta: float = 0.5,
) -> torch.Tensor:
    mse_term = F.mse_loss(current_logits, past_logits)
    ce_term = F.cross_entropy(current_logits, labels)
    return (alpha * mse_term + beta * ce_term).unsqueeze(0)

der_calc = der_dark_replay_loss(golden["current_logits"], golden["past_logits"], golden["labels"], alpha=0.5, beta=0.5)
diff_der = torch.abs(der_calc - golden["der_loss_ref"]).item()
assert diff_der < 1e-5

# Anti-Pattern / Misuse Failure: Negative path integral in SI without non-negative clamping
raw_path_integral_negative = torch.tensor([-5.0, 2.0])
delta = torch.tensor([1.0, 1.0])
# Without clamp, negative path integrals reward movement in forgotten directions!
bad_omega = raw_path_integral_negative / (delta ** 2 + 0.1)
assert bad_omega[0].item() < 0.0, "Anti-Pattern: Unclamped path integral produces negative importance"
# Correct SI clamps negative values to 0:
good_omega = torch.clamp(raw_path_integral_negative, min=0.0) / (delta ** 2 + 0.1)
assert good_omega[0].item() == 0.0, "SI Failure: Clamping must eliminate negative importance"

print("=" * 65)
print(f"{'Operator':<32} | {'Max Abs Error':<15} | Status")
print("-" * 65)
print(f"{'EWC Quadratic Penalty':<32} | {diff_ewc:<15.2e} | PASSED")
print(f"{'Synaptic Intelligence Omega':<32} | {diff_si:<15.2e} | PASSED")
print(f"{'LwF Distillation Loss':<32} | {diff_lwf:<15.2e} | PASSED")
print(f"{'DER++ Dark Replay Loss':<32} | {diff_der:<15.2e} | PASSED")
print("=" * 65)

## 8. Summary, Key Takeaways & Cross-Domain Generalisation

- [x] **EWC**: Protects parameters in directions of high Fisher Information curvature to mitigate catastrophic forgetting.
- [x] **Synaptic Intelligence**: Online path integral computation allows tracking importance without separate offline Fisher evaluation.
- [x] **LwF**: Distills predictions on current data using frozen historical heads without accessing past raw samples.
- [x] **DER++**: Replaying un-normalized dark logits regularizes the entire output manifold, drastically outperforming hard-label rehearsal.

**Cross-Domain Generalisation**:
Continual learning mechanisms prevent catastrophic forgetting in autonomous driving (adapting to new weather conditions while retaining prior driving rules), robotics (learning new manipulation skills without forgetting navigation), and automated medical diagnosis (incorporating newly discovered disease classes).

See curriculum index: [INDEX_BY_GOAL](../../../INDEX_BY_GOAL.md).